# Notebook 01: Extração e Pré-processamento
Este notebook filtra os idosos do banco mestre e cria os 4 subgrupos clínicos.

In [1]:
# Importa o módulo sqlite3, que permite conversar com bancos de dados SQLite locais.
import sqlite3
# Importa o pandas, a biblioteca principal para manipulação de tabelas (DataFrames) no Python.
import pandas as pd
# Importa os/sys, para lidar com caminhos de pastas e arquivos.
import os, sys

# Define o caminho onde os bancos de dados estão salvos.
PASTA_DB = '../data/database/'
# Junta o caminho da pasta com o nome do arquivo do banco mestre gigante.
DB_MASTER = os.path.join(PASTA_DB, 'pns_master_formatado.db')

# ── Bootstrap + validação do mestre (os .db são gitignored / NÃO versionados) ──
# O banco-mestre (~200 MB) raramente está numa cópia nova do projeto. Este NB01 só
# reconstrói os 4 grupos A PARTIR do mestre; se ele não existir ou for inválido
# (ex.: arquivo-fantasma de 0 byte), garantimos os 4 bancos derivados pelo bootstrap
# (a partir dos CSVs-semente) e PULAMOS as células de extração do mestre.
for _p in (os.getcwd(), os.path.join(os.getcwd(), 'notebooks')):
    if os.path.isfile(os.path.join(_p, 'preparar_bancos.py')) and _p not in sys.path:
        sys.path.insert(0, _p)
from preparar_bancos import garantir_bancos, _tabelas

# Mestre é válido só se o arquivo existe, NÃO está vazio e tem a tabela pns_completa.
# (NUNCA conectar a um caminho inexistente: o sqlite criaria o arquivo-fantasma de 0 byte.)
MESTRE_OK = os.path.exists(DB_MASTER) and os.path.getsize(DB_MASTER) > 0
if MESTRE_OK:
    MESTRE_OK = 'pns_completa' in _tabelas(DB_MASTER)
print(f"Banco mestre válido (com a tabela pns_completa)? {MESTRE_OK}")

if not MESTRE_OK:
    print("→ Mestre ausente/inválido. Garantindo os 4 bancos derivados via bootstrap (CSVs-semente)…")
    garantir_bancos()   # regenera só o que faltar; também remove o arquivo-fantasma do mestre
    print("→ Pronto. As células de extração do mestre serão PULADAS. Pode seguir para o NB02.")


Banco mestre válido (com a tabela pns_completa)? False
→ Mestre ausente/inválido. Garantindo os 4 bancos derivados via bootstrap (CSVs-semente)…
→ Pronto. As células de extração do mestre serão PULADAS. Pode seguir para o NB02.


In [2]:
# Cria uma lista com o código das 14 perguntas sobre doenças crônicas do questionário da PNS (Módulo Q).
# O Q079 é a variável principal do nosso estudo: diagnóstico de Artrite ou Reumatismo.
variaveis_doencas = [
    'Q00201', # Hipertensão
    'Q03001', # Diabetes
    'Q060',   # Colesterol
    'Q06306', # Doença do coração
    'Q068',   # AVC
    'Q074',   # Asma
    'Q079',   # Artrite/Reumatismo (Nossa Variável Alvo)
    'Q088',   # DORT
    'Q092',   # Depressão
    'Q11006', # Outra doença mental
    'Q11604', # Doença crônica no pulmão
    'Q120',   # Câncer
    'Q124',   # Insuficiência renal crônica
    'Q128'    # Outra doença crônica
]

# Cria um comando (string) SQL para buscar apenas as pessoas com 60 anos ou mais.
# Como a idade (C008) está salva como texto no banco, usamos CAST para forçar o SQLite a ler como número inteiro (INTEGER).
filtro_idosos = "CAST(C008 AS INTEGER) >= 60"


In [3]:
# Só executa a extração a partir do mestre se ele estiver disponível e válido.
if MESTRE_OK:
    print("Extraindo Idosos Geral...")                                # avisa que vai começar
    conn_master = sqlite3.connect(DB_MASTER)                          # abre o banco mestre
    query_geral = f"SELECT * FROM pns_completa WHERE {filtro_idosos}" # idade >= 60
    df_idosos = pd.read_sql_query(query_geral, conn_master)           # traz os idosos p/ a RAM
    conn_master.close()                                              # fecha a conexão
    print(f"Total de Idosos encontrados: {len(df_idosos)}")
    conn_geral = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_geral.db'))  # novo banco
    df_idosos.to_sql('pns_idosos', conn_geral, if_exists='replace', index=False)
    conn_geral.close()
    print("Banco 'idosos_geral.db' salvo com sucesso.")
else:
    print("⏭️  NB01 pulou a extração do mestre — os 4 bancos já foram garantidos pelo bootstrap.")


⏭️  NB01 pulou a extração do mestre — os 4 bancos já foram garantidos pelo bootstrap.


## Critério de classificação Sim/Não (diagnóstico autorreferido)

As 14 variáveis do Módulo Q (doenças crônicas) assumem no banco apenas **`Sim`**, **`Não`** ou
**`NULL`** (não-aplicável / morador não selecionado) — não há `Não sabe`/`Ignorado` como texto.

**Convenção adotada:**
- **Artrite** = `Q079 = Sim` (grupo de casos).
- **Saudável** exige **`Não` explícito nas 14** doenças. `NULL` (ou qualquer valor que não seja
  exatamente `Não`) **não** conta como `Não`, então esses moradores **não** entram em Saudável
  (critério conservador).
- **Limitação p/ o artigo:** diagnóstico **autorreferido** — ausência de afirmação de diagnóstico
  é tratada como ausência da doença.


In [4]:
# == Classificação Sim/Não das variáveis de doença (módulo Q) ====================
# (Só roda quando o mestre está disponível — depende de df_idosos da célula anterior.)
def eh_nao(val):
    if pd.isna(val): return False                          # NULL não é "Não"
    return str(val).strip().lower() in ('não', 'nao', '2')  # só 'Não' explícito (ou código 2)

def eh_sim(val):
    if pd.isna(val): return False
    return str(val).strip().lower() in ('sim', '1')         # só 'Sim' explícito (ou código 1)

if MESTRE_OK:
    mask_artrite = df_idosos['Q079'].apply(eh_sim)                       # quem tem artrite
    doencas_sem_artrite = [v for v in variaveis_doencas if v != 'Q079']  # 13 outras doenças
    try:
        mask_outras_doencas = df_idosos[doencas_sem_artrite].map(eh_sim).any(axis=1)
    except AttributeError:
        mask_outras_doencas = df_idosos[doencas_sem_artrite].applymap(eh_sim).any(axis=1)
    try:
        mask_saudavel = df_idosos[variaveis_doencas].map(eh_nao).all(axis=1)
    except AttributeError:
        mask_saudavel = df_idosos[variaveis_doencas].applymap(eh_nao).all(axis=1)
    df_artrite = df_idosos[mask_artrite]                                 # com artrite (+comorb.)
    df_artrite_puro = df_idosos[mask_artrite & ~mask_outras_doencas]     # artrite pura
    df_saudaveis = df_idosos[mask_saudavel]                              # 100% saudáveis
    print(f"Idosos com Artrite: {len(df_artrite)}")
    print(f"Idosos com Artrite Pura: {len(df_artrite_puro)}")
    print(f"Idosos Saudáveis: {len(df_saudaveis)}")
else:
    print("⏭️  Classificação pulada (sem mestre).")


⏭️  Classificação pulada (sem mestre).


In [5]:
# Salva os 3 subgrupos derivados (só quando o mestre foi processado nesta sessão).
if MESTRE_OK:
    conn_artrite = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite.db'))
    df_artrite.to_sql('pns_idosos', conn_artrite, if_exists='replace', index=False)
    conn_artrite.close()
    conn_artrite_puro = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite_puro.db'))
    df_artrite_puro.to_sql('pns_idosos', conn_artrite_puro, if_exists='replace', index=False)
    conn_artrite_puro.close()
    conn_saudaveis = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_saudaveis.db'))
    df_saudaveis.to_sql('pns_idosos', conn_saudaveis, if_exists='replace', index=False)
    conn_saudaveis.close()
    print("Todos os bancos foram recriados e salvos com sucesso!")
else:
    print("⏭️  Gravação pulada (sem mestre) — os bancos já vieram do bootstrap.")


⏭️  Gravação pulada (sem mestre) — os bancos já vieram do bootstrap.
